# Fine-tune DINOv2-Large + UperNet (Pruned Classes + Multi-Axis Training)

Thin orchestration notebook for Databricks. All reusable logic lives in
`histological_image_analysis.training` (installed as a wheel on the cluster).

**Run 8 — Data-centric improvements (Step 12, Step 4):**
- **Class pruning:** Remove 657 zero-pixel classes from the output head. Train on
  ~671 classes that actually have training pixels. Eliminates softmax dilution,
  reduces gradient noise, halves decode head VRAM.
- **Multi-axis training:** Add sagittal (axis 2) and axial (axis 1) slices to
  training data. Val/test remain coronal-only (identical to Run 5) to maintain
  comparability and avoid cross-orientation leakage.
- **Baseline augmentation:** flip + rot15° + jitter (Run 5 config). No rot90,
  elastic, or blur (Run 7 showed these regress performance).
- **Standard CE loss:** No Dice component (Run 6 showed weighted loss regresses).

**Motivation:** The model peaked at 68.8% mIoU (Run 5) with 1,016 coronal
training samples across 1,328 classes. Three subsequent experiments failed:
- Run 6 (weighted Dice+CE): −1.6% mIoU
- Run 7 (extended augmentation): −6.5% mIoU
- TTA (6-variant eval): −24% mIoU (catastrophic, model lacks rotational equivariance)

Root cause: data scarcity (657 classes with zero training pixels). This run
addresses the root cause directly.

**Memory optimizations (same as Run 5):**
- `per_device_train_batch_size=2` + `gradient_accumulation_steps=2` (effective batch=4)
- Gradient checkpointing on backbone only with `use_reentrant=False`
- `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True`

**Cluster:** Single GPU (1x L40S 48 GB).

In [0]:
# Cell 0 — Install project wheel from DBFS
# The wheel is uploaded by `make deploy-wheel` and contains all reusable
# training pipeline code (ontology, slicer, dataset, training modules).

%pip install /dbfs/FileStore/wheels/histological_image_analysis-0.1.0-py3-none-any.whl
dbutils.library.restartPython()

Processing /dbfs/FileStore/wheels/histological_image_analysis-0.1.0-py3-none-any.whl
  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
  Using cached pynrrd-1.1.3-py3-none-any.whl.metadata (5.4 kB)
  Using cached svgpathtools-1.7.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached svgwrite-1.4.3-py3-none-any.whl.metadata (8.8 kB)
Using cached accelerate-1.13.0-py3-none-any.whl (383 kB)
Using cached pynrrd-1.1.3-py3-none-any.whl (23 kB)
Using cached svgpathtools-1.7.2-py2.py3-none-any.whl (68 kB)
Using cached svgwrite-1.4.3-py3-none-any.whl (67 kB)
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.5.2
    Not uninstalling accelerate at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-7f2a5ce7-f09d-4536-b2f4-bae820bb254f
    Can't uninstall 'accelerate'. No files were found to uninstall.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restar

In [0]:
# Cell 1 — Configuration
# All environment-specific paths and hyperparameters live here.

import os
import mlflow

# ---------- CUDA memory optimization ----------
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ---------- MLflow setup ----------
os.environ["MLFLOW_ENABLE_SYSTEM_METRICS_LOGGING"] = "true"

EXPERIMENT_NAME = "/Users/noel.nosse@grainger.com/histology-brain-segmentation"
try:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
except Exception:
    experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
mlflow.set_experiment(experiment_id=experiment_id)
print(f"MLflow experiment: {EXPERIMENT_NAME} (ID: {experiment_id})")

# ---------- Databricks paths ----------
WORKSPACE_BASE = "/Workspace/Users/noel.nosse@grainger.com/visual-model-ft/histology"
ONTOLOGY_PATH = f"{WORKSPACE_BASE}/ontology/structure_graph_1.json"
ANNOTATION_10_PATH = f"{WORKSPACE_BASE}/ccfv3/annotation_10.nrrd"
NISSL_10_PATH = "/dbfs/FileStore/allen_brain_data/ccfv3/ara_nissl_10.nrrd"

# ---------- JFrog / HuggingFace model ----------
HF_ENDPOINT = os.environ.get(
    "HF_ENDPOINT",
    "https://graingerreadonly.jfrog.io/artifactory/api/huggingfaceml/huggingfaceml-remote",
)
MODEL_REPO_ID = "facebook/dinov2-large"
MODEL_CACHE_DIR = "/tmp/dinov2-large"

# ---------- Mapping ----------
MAPPING_TYPE = "full_pruned"  # full mapping filtered to present classes

# ---------- Training hyperparameters ----------
# Same as Run 5 (unfrozen baseline) except:
# - NUM_LABELS: ~671 (from class pruning, computed in Cell 3)
# - Training samples: ~2,000-2,600 (from multi-axis slicing)
# - augmentation_preset="baseline" (flip + rot15° + jitter)
BACKBONE_LR = 1e-5
HEAD_LR = 1e-4
NUM_UNFROZEN_BLOCKS = 4
BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 2

HYPERPARAMS = {
    "mapping_type": MAPPING_TYPE,
    "crop_size": 518,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "effective_batch_size": BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
    "backbone_lr": BACKBONE_LR,
    "head_lr": HEAD_LR,
    "num_unfrozen_blocks": NUM_UNFROZEN_BLOCKS,
    "num_epochs": 100,
    "freeze_backbone": False,
    "gradient_checkpointing": "backbone_only_non_reentrant",
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "split_strategy": "interleaved",
    "model": MODEL_REPO_ID,
    "dataset": "CCFv3 ara_nissl_10",
    "augmentation": "baseline: flip_50pct, rot15_always, jitter_always",
    "class_pruning": True,
    "multi_axis_training": True,
}

CROP_SIZE = HYPERPARAMS["crop_size"]
NUM_EPOCHS = HYPERPARAMS["num_epochs"]
SPLIT_STRATEGY = HYPERPARAMS["split_strategy"]

# ---------- Output ----------
OUTPUT_DIR = "/tmp/checkpoints/pruned-multiaxis"
FINAL_MODEL_DIR = "/dbfs/FileStore/allen_brain_data/models/pruned-multiaxis"

print(f"Ontology:           {ONTOLOGY_PATH}")
print(f"Nissl 10\u00b5m:         {NISSL_10_PATH}")
print(f"HF endpoint:        {HF_ENDPOINT}")
print(f"Mapping:            {MAPPING_TYPE}")
print(f"Augmentation:       baseline (flip + rot15\u00b0 + jitter)")
print(f"Class pruning:      True (remove zero-pixel classes)")
print(f"Multi-axis train:   True (coronal + axial + sagittal)")
print(f"Batch size:         {BATCH_SIZE} (x{GRADIENT_ACCUMULATION_STEPS} grad accum = effective {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS})")
print(f"Backbone LR:        {BACKBONE_LR}")
print(f"Head LR:            {HEAD_LR}")
print(f"Unfrozen blocks:    last {NUM_UNFROZEN_BLOCKS} (blocks {24 - NUM_UNFROZEN_BLOCKS}-23)")
print(f"Epochs:             {NUM_EPOCHS} (with early stopping)")
print(f"Checkpoints:        {OUTPUT_DIR}")
print(f"Final model:        {FINAL_MODEL_DIR}")

MLflow experiment: /Users/noel.nosse@grainger.com/histology-brain-segmentation (ID: 1345391216675532)
Ontology:           /Workspace/Users/noel.nosse@grainger.com/visual-model-ft/histology/ontology/structure_graph_1.json
Nissl 10µm:         /dbfs/FileStore/allen_brain_data/ccfv3/ara_nissl_10.nrrd
HF endpoint:        https://graingerreadonly.jfrog.io/artifactory/api/huggingfaceml/huggingfaceml-remote
Mapping:            full_pruned
Augmentation:       baseline (flip + rot15° + jitter)
Class pruning:      True (remove zero-pixel classes)
Multi-axis train:   True (coronal + axial + sagittal)
Batch size:         2 (x2 grad accum = effective 4)
Backbone LR:        1e-05
Head LR:            0.0001
Unfrozen blocks:    last 4 (blocks 20-23)
Epochs:             100 (with early stopping)
Checkpoints:        /tmp/checkpoints/pruned-multiaxis
Final model:        /dbfs/FileStore/allen_brain_data/models/pruned-multiaxis


In [0]:
# Cell 2 — Download model weights from JFrog Artifactory mirror

import time
from huggingface_hub import snapshot_download

print(f"Downloading {MODEL_REPO_ID} from Artifactory...")
for attempt in range(1, 4):
    try:
        model_path = snapshot_download(
            repo_id=MODEL_REPO_ID,
            endpoint=HF_ENDPOINT,
            etag_timeout=86400,
            cache_dir=MODEL_CACHE_DIR,
        )
        break
    except Exception as e:
        if attempt < 3:
            wait = 2 ** attempt
            print(f"  Attempt {attempt} failed ({e.__class__.__name__}), retrying in {wait}s...")
            time.sleep(wait)
        else:
            raise

print(f"Model downloaded to: {model_path}")

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

(…)6e6d21ea90829a6652ed6f75ec47/config.json:   0%|          | 0.00/549 [00:00<?, ?B/s]

(…)a6652ed6f75ec47/preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

(…)496e6d21ea90829a6652ed6f75ec47/README.md:   0%|          | 0.00/3.03k [00:00<?, ?B/s]

(…)d21ea90829a6652ed6f75ec47/.gitattributes:   0%|          | 0.00/1.52k [00:00<?, ?B/s]

(…)ea90829a6652ed6f75ec47/model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

(…)ea90829a6652ed6f75ec47/pytorch_model.bin:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

Model downloaded to: /tmp/dinov2-large/models--facebook--dinov2-large/snapshots/a18992645e496e6d21ea90829a6652ed6f75ec47


In [0]:
# Cell 3 — Build pruned mapping + multi-axis datasets
#
# Pipeline:
#   1. Build full mapping (all 1,328 classes)
#   2. Scan coronal training slices to find which classes have >0 pixels
#   3. Build pruned mapping (only present classes, contiguous IDs)
#   4. Create multi-axis training dataset + coronal-only val dataset
#
# Class pruning removes 657 zero-pixel classes from the output head.
# Multi-axis adds sagittal + axial slices to training only.
# Val/test remain coronal-only (identical to Run 5).

import numpy as np
from histological_image_analysis.ontology import OntologyMapper
from histological_image_analysis.ccfv3_slicer import CCFv3Slicer
from histological_image_analysis.dataset import BrainSegmentationDataset

# 1. Load ontology and build full mapping
mapper = OntologyMapper(ONTOLOGY_PATH)
full_mapping = mapper.build_full_mapping()
full_num_labels = mapper.get_num_labels(full_mapping)
print(f"Full mapping: {full_num_labels} classes (including background)")

# 2. Load CCFv3 volumes
slicer = CCFv3Slicer(
    image_path=NISSL_10_PATH,
    annotation_path=ANNOTATION_10_PATH,
    ontology_mapper=mapper,
)
slicer.load_volumes()
print(f"Loaded volume: shape={slicer.image_volume.shape}")

# 3. Scan training slices to find present class IDs
# Use coronal training slices only (same split as Run 5) for scanning.
# Multi-axis slices will add more data but the class presence is determined
# by coronal slices to keep the mapping deterministic and comparable.
print("\nScanning training slices for present classes...")
present_class_ids = set()
n_scanned = 0
for img, class_mask in slicer.iter_slices(
    "train", full_mapping, split_strategy=SPLIT_STRATEGY,
):
    present_class_ids.update(np.unique(class_mask).tolist())
    n_scanned += 1

# Remove background (0) from present set for reporting, but it's always included
present_nonbg = present_class_ids - {0}
absent = full_num_labels - 1 - len(present_nonbg)  # -1 for background
print(f"Scanned {n_scanned} coronal training slices")
print(f"Present classes (excl. background): {len(present_nonbg)}")
print(f"Absent classes (zero training pixels): {absent}")

# 4. Build pruned mapping
mapping = mapper.build_present_mapping(full_mapping, present_class_ids)
NUM_LABELS = mapper.get_num_labels(mapping)
class_names = mapper.get_class_names(mapping)
HYPERPARAMS["num_labels"] = NUM_LABELS
HYPERPARAMS["present_classes"] = len(present_nonbg)
HYPERPARAMS["absent_classes"] = absent
print(f"\nPruned mapping: {NUM_LABELS} classes (including background)")
print(f"Reduction: {full_num_labels} \u2192 {NUM_LABELS} ({full_num_labels - NUM_LABELS} classes removed)")

# 5. Get split info (coronal only, for reporting)
splits = slicer.get_split_indices(
    train_frac=0.8, val_frac=0.1, split_strategy=SPLIT_STRATEGY,
)
print(f"\nCoronal splits: Train={len(splits['train'])} | Val={len(splits['val'])} | Test={len(splits['test'])}")

# 6. Create multi-axis training dataset (coronal + axial + sagittal)
train_ds = BrainSegmentationDataset(
    slicer=slicer, split="train", mapping=mapping,
    crop_size=CROP_SIZE, augment=True, split_strategy=SPLIT_STRATEGY,
    augmentation_preset="baseline",
    multi_axis=True,  # <-- adds axial + sagittal slices
)
val_ds = BrainSegmentationDataset(
    slicer=slicer, split="val", mapping=mapping,
    crop_size=CROP_SIZE, augment=False, split_strategy=SPLIT_STRATEGY,
    # multi_axis=False (default) — val stays coronal-only
)
HYPERPARAMS["train_samples"] = len(train_ds)
HYPERPARAMS["val_samples"] = len(val_ds)
print(f"\nTrain samples: {len(train_ds)} (coronal + axial + sagittal)")
print(f"Val samples:   {len(val_ds)} (coronal only)")
print(f"Data multiplier: {len(train_ds) / len(splits['train']):.1f}x vs coronal-only")

# Sanity check
sample = train_ds[0]
print(f"\npixel_values: {sample['pixel_values'].shape}, labels: {sample['labels'].shape}")
print(f"Unique classes in sample: {len(set(sample['labels'].numpy().ravel()))}")
print(f"Max class ID in sample: {sample['labels'].max().item()} (should be < {NUM_LABELS})")
assert sample['labels'].max().item() < NUM_LABELS, "Class ID exceeds pruned NUM_LABELS!"

Full mapping: 1328 classes (including background)
Loaded volume: shape=(1320, 800, 1140)

Scanning training slices for present classes...
Scanned 1016 coronal training slices
Present classes (excl. background): 672
Absent classes (zero training pixels): 655

Pruned mapping: 673 classes (including background)
Reduction: 1328 → 673 (655 classes removed)

Coronal splits: Train=1016 | Val=127 | Test=127

Train samples: 2649 (coronal + axial + sagittal)
Val samples:   127 (coronal only)
Data multiplier: 2.6x vs coronal-only

pixel_values: torch.Size([3, 518, 518]), labels: torch.Size([518, 518])
Unique classes in sample: 4
Max class ID in sample: 520 (should be < 673)


In [0]:
# Cell 4 — Create model with partially unfrozen backbone
#
# Same strategy as Run 5: unfreeze last 4 blocks + layernorm.
# Key difference: NUM_LABELS is ~671 instead of 1,328.
# This halves the decode head size, reducing VRAM and compute.

import torch
from histological_image_analysis.training import create_model

model = create_model(
    num_labels=NUM_LABELS,
    freeze_backbone=False,
    pretrained_backbone_path=model_path,
)

first_frozen_block = 24 - NUM_UNFROZEN_BLOCKS

# Freeze embeddings
for param in model.backbone.embeddings.parameters():
    param.requires_grad = False

# Freeze encoder blocks 0 through (first_frozen_block - 1)
for i in range(first_frozen_block):
    for param in model.backbone.encoder.layer[i].parameters():
        param.requires_grad = False

# Enable gradient checkpointing on backbone only
# use_reentrant=False is REQUIRED at the frozen/unfrozen boundary
model.backbone.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)
print("Gradient checkpointing: ENABLED (backbone only, use_reentrant=False)")

# Verify: count trainable params by component
backbone_frozen = sum(p.numel() for p in model.backbone.parameters() if not p.requires_grad)
backbone_trainable = sum(p.numel() for p in model.backbone.parameters() if p.requires_grad)
head_trainable = sum(
    p.numel() for n, p in model.named_parameters()
    if p.requires_grad and "backbone" not in n
)
total = sum(p.numel() for p in model.parameters())

print(f"Backbone frozen:    {backbone_frozen:>12,} params")
print(f"Backbone trainable: {backbone_trainable:>12,} params (blocks {first_frozen_block}-23 + layernorm)")
print(f"Head trainable:     {head_trainable:>12,} params (decode_head + auxiliary_head)")
print(f"Total trainable:    {backbone_trainable + head_trainable:>12,} / {total:,} ({(backbone_trainable + head_trainable) / total * 100:.1f}%)")
print(f"\nNote: Head is smaller than Run 5 due to pruned output ({NUM_LABELS} vs 1,328 classes)")

# Sanity check forward pass
model.eval()
with torch.no_grad():
    dummy = torch.randn(1, 3, CROP_SIZE, CROP_SIZE)
    if torch.cuda.is_available():
        model = model.cuda()
        dummy = dummy.cuda()
    out = model(pixel_values=dummy)
    print(f"\nLogits shape: {out.logits.shape}")
    print(f"Expected: (1, {NUM_LABELS}, {CROP_SIZE}, {CROP_SIZE})")
    logits_mb = out.logits.element_size() * out.logits.nelement() / 1e6
    print(f"Logits memory (single sample): {logits_mb:.1f} MB")
    assert out.logits.shape == (1, NUM_LABELS, CROP_SIZE, CROP_SIZE)

# Free GPU memory from sanity check
del out, dummy
torch.cuda.empty_cache()
model = model.cpu()
torch.cuda.empty_cache()

import subprocess
print("\nGPU memory after cleanup:")
print(subprocess.check_output(["nvidia-smi", "--query-gpu=memory.used,memory.free,memory.total",
                                "--format=csv,noheader,nounits"]).decode().strip(), "MiB (used, free, total)")
print("Forward pass OK")

2026-03-15 17:42:08.384163: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773596528.399787    9358 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773596528.404630    9358 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773596528.417863    9358 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773596528.417877    9358 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773596528.417879    9358 computation_placer.cc:177] computation placer alr

[2026-03-15 17:42:10,632] [INFO] [real_accelerator.py:239:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -lcufile: No such file or directory
collect2: error: ld returned 1 exit status


Gradient checkpointing: ENABLED (backbone only, use_reentrant=False)
Backbone frozen:     253,973,504 params
Backbone trainable:   50,395,136 params (blocks 20-23 + layernorm)
Head trainable:       37,231,170 params (decode_head + auxiliary_head)
Total trainable:      87,626,306 / 341,599,810 (25.7%)

Note: Head is smaller than Run 5 due to pruned output (673 vs 1,328 classes)

Logits shape: torch.Size([1, 673, 518, 518])
Expected: (1, 673, 518, 518)
Logits memory (single sample): 722.3 MB

GPU memory after cleanup:
10218, 35377, 46068 MiB (used, free, total)
Forward pass OK


In [ ]:
# Cell 5 — Train with differential learning rate
#
# Same optimizer setup as Run 5: AdamW with 2 param groups.
# Key differences:
# - More training samples (multi-axis) → more steps per epoch
# - Smaller output head (pruned classes) → less VRAM per step

import torch
from datetime import datetime
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from histological_image_analysis.training import (
    get_training_args,
    make_compute_metrics,
    preprocess_logits_for_metrics,
)
from transformers import Trainer

mlflow.start_run(run_name=f"pruned-multiaxis-{NUM_LABELS}class-{datetime.now().strftime('%Y%m%d-%H%M')}")
mlflow.log_params(HYPERPARAMS)

training_args = get_training_args(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=HEAD_LR,
    fp16=torch.cuda.is_available(),
    report_to="mlflow",
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    ddp_find_unused_parameters=True,
    dataloader_drop_last=True,  # REQUIRED: multi-axis gives 2,649 samples (odd).
    # Without this, last batch has 1 sample → UperNet PSP BatchNorm crashes on [1,512,1,1].
)

backbone_params = [
    p for n, p in model.named_parameters()
    if p.requires_grad and "backbone" in n
]
head_params = [
    p for n, p in model.named_parameters()
    if p.requires_grad and "backbone" not in n
]

optimizer = AdamW(
    [
        {"params": backbone_params, "lr": BACKBONE_LR},
        {"params": head_params, "lr": HEAD_LR},
    ],
    weight_decay=HYPERPARAMS["weight_decay"],
)

num_training_steps = (len(train_ds) // (BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)) * NUM_EPOCHS
num_warmup_steps = int(num_training_steps * HYPERPARAMS["warmup_ratio"])

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps,
)

print(f"Optimizer: AdamW with 2 param groups")
print(f"  Backbone ({len(backbone_params)} tensors): lr={BACKBONE_LR}")
print(f"  Head ({len(head_params)} tensors):     lr={HEAD_LR}")
print(f"  Batch size: {BATCH_SIZE} x {GRADIENT_ACCUMULATION_STEPS} grad accum = effective {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"  Training samples: {len(train_ds)} (multi-axis, drop_last=True)")
print(f"  Total training steps: {num_training_steps}")
print(f"  Steps per epoch: {len(train_ds) // (BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)}")
print(f"  Warmup steps: {num_warmup_steps}")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=make_compute_metrics(NUM_LABELS),
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,
    optimizers=(optimizer, scheduler),
)

trainer.train()

In [0]:
# Cell 6 — Evaluate + Visualize
#
# Val set is coronal-only (identical to Run 5) for direct comparison.
# mIoU is computed over the pruned class set (~671 classes).

import matplotlib.pyplot as plt
import numpy as np

eval_results = trainer.evaluate()
print("Evaluation results:")
for k, v in sorted(eval_results.items()):
    if isinstance(v, float) and not k.startswith("eval_iou_class_"):
        print(f"  {k}: {v:.4f}")

mlflow.log_metrics({
    "final_mean_iou": eval_results.get("eval_mean_iou", 0.0),
    "final_overall_accuracy": eval_results.get("eval_overall_accuracy", 0.0),
})

# Per-class IoU summary
class_ious = {}
for cls in range(NUM_LABELS):
    iou_key = f"eval_iou_class_{cls}"
    iou = eval_results.get(iou_key, float('nan'))
    if not np.isnan(iou):
        class_ious[cls] = iou

sorted_ious = sorted(class_ious.items(), key=lambda x: x[1], reverse=True)
print(f"\nTop 10 classes by IoU:")
for cls, iou in sorted_ious[:10]:
    name = class_names[cls] if cls < len(class_names) else f"class_{cls}"
    print(f"  {cls:4d} ({name:40s}): {iou:.4f}")

print(f"\nBottom 10 classes by IoU:")
for cls, iou in sorted_ious[-10:]:
    name = class_names[cls] if cls < len(class_names) else f"class_{cls}"
    print(f"  {cls:4d} ({name:40s}): {iou:.4f}")

print(f"\nClasses with non-NaN IoU: {len(class_ious)} / {NUM_LABELS}")

# Compare with previous runs
# NOTE: Run 5 mIoU (68.8%) was computed over 1,328-class mapping.
# This run uses ~671 classes. mIoU is not directly comparable because
# the 657 removed classes all had 0.0 IoU, dragging Run 5's mean down.
# For fair comparison, look at overall accuracy (comparable across mappings)
# and the per-class IoU of structures that exist in both mappings.
current_miou = eval_results.get('eval_mean_iou', 0.0)
current_acc = eval_results.get('eval_overall_accuracy', 0.0)
print(f"\n--- Comparison (note: mIoU computed over different class sets) ---")
print(f"Run 5 (unfrozen, 1,328 classes): mIoU=68.8%, accuracy=92.5%")
print(f"Run 8 (pruned, {NUM_LABELS} classes):  mIoU={current_miou:.1%}, accuracy={current_acc:.1%}")
print(f"")
print(f"Run 5 had 503/1,328 valid classes. 825 classes had NaN IoU (0 val pixels).")
print(f"Run 8 has {len(class_ious)}/{NUM_LABELS} valid classes.")
print(f"")
print(f"Overall accuracy is the most comparable metric (same val set, same pixel predictions).")

# Visualize predictions vs ground truth
model.eval()
fig, axes = plt.subplots(3, 3, figsize=(15, 15))

for i in range(3):
    sample = val_ds[i * (len(val_ds) // 3)]
    pixel_values = sample["pixel_values"].unsqueeze(0)
    labels = sample["labels"].numpy()

    with torch.no_grad():
        if torch.cuda.is_available():
            pixel_values = pixel_values.cuda()
        logits = model(pixel_values=pixel_values).logits
        preds = logits.argmax(dim=1).squeeze().cpu().numpy()

    img = sample["pixel_values"].permute(1, 2, 0).numpy()
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)

    axes[i, 0].imshow(img)
    axes[i, 0].set_title("Input")
    axes[i, 1].imshow(labels, cmap="nipy_spectral")
    axes[i, 1].set_title("Ground Truth")
    axes[i, 2].imshow(preds, cmap="nipy_spectral")
    axes[i, 2].set_title("Prediction")

for ax in axes.ravel():
    ax.axis("off")

plt.suptitle(
    f"Pruned + multi-axis | {NUM_LABELS}-class | "
    f"mIoU={current_miou:.3f} | acc={current_acc:.3f}"
)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 7 — Save final model + processor + log to MLflow + close run
#
# CRITICAL: UperNetForSemanticSegmentation is NOT compatible with
# mlflow.transformers.log_model() — it is not a Pipeline and not in the
# AutoModel registry. Neither passing the model object nor a path works.
# Use mlflow.log_artifacts() to log the raw checkpoint files.
# Load later with: UperNetForSemanticSegmentation.from_pretrained(path)

import os
import json
from transformers import AutoImageProcessor

os.makedirs(FINAL_MODEL_DIR, exist_ok=True)

# 1. Set id2label and label2id on model config (create_model doesn't set these)
model.config.id2label = {i: name for i, name in enumerate(class_names)}
model.config.label2id = {name: i for i, name in enumerate(class_names)}
print(f"Set id2label/label2id: {NUM_LABELS} classes")

# 2. Save model to DBFS as backup
trainer.save_model(FINAL_MODEL_DIR)
print(f"Model saved to: {FINAL_MODEL_DIR}")

# 3. Save image processor (trainer.save_model does NOT save it)
processor = AutoImageProcessor.from_pretrained(model_path)
processor.save_pretrained(FINAL_MODEL_DIR)
print(f"Image processor saved to: {FINAL_MODEL_DIR}")

# 4. Save pruned mapping metadata for inference-time reconstruction
mapping_metadata = {
    "num_labels": NUM_LABELS,
    "full_num_labels": full_num_labels,
    "present_classes": len(present_nonbg),
    "absent_classes": absent,
    "class_names": class_names,
}
metadata_path = os.path.join(FINAL_MODEL_DIR, "pruned_mapping_metadata.json")
with open(metadata_path, "w") as f:
    json.dump(mapping_metadata, f, indent=2)
print(f"Mapping metadata saved to: {metadata_path}")

# 5. Log to MLflow — raw file artifacts (UperNet is incompatible with mlflow.transformers)
mlflow.log_artifacts(FINAL_MODEL_DIR, artifact_path="model")
print("Model artifacts logged to MLflow under 'model/'")

# 6. Log mapping metadata as separate artifact
mlflow.log_artifact(metadata_path, artifact_path="metadata")
print("Mapping metadata logged as MLflow artifact")

# 7. Close MLflow run
mlflow.end_run()
print("MLflow run closed.")